In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("2  powerplant_data (1).csv")
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [3]:
# at->temp
# v->vaccum
#ap->pressure
# rh->humidity

# pe->produced energy


In [4]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [5]:
x=df.drop("PE",axis=1)
y= df["PE"]


In [6]:
x.head()


,AT,V,AP,RH
0,8.34,40.77,1010.84,90.01
1,23.64,58.49,1011.40,74.20
2,29.74,56.90,1007.15,41.91
3,19.07,49.69,1007.22,76.79
4,11.80,40.66,1017.13,97.20


In [7]:
y.head()

0    480.48
1    445.75
2    438.76
3    453.09
4    464.43
Name: PE, dtype: float64

In [8]:
# converting->data to tensors

In [9]:
# split data
from sklearn.model_selection import train_test_split


In [10]:
x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=42,test_size=0.2)


In [11]:
df.shape

(9568, 5)

In [12]:
from sklearn.preprocessing import StandardScaler


In [13]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_train)


In [14]:
x_train_scaled

array([[ 0.74805289,  0.72006931, -0.32660017, -0.49711722],
       [ 0.86181948,  1.26515721, -0.98521113,  0.8181501 ],
       [ 0.93409473,  1.52314975,  0.32523844,  0.80167494],
       ...,
       [-0.22097078, -0.834965  ,  0.36756563, -0.83554456],
       [ 0.94747903,  1.14245344, -0.41971997, -0.45455637],
       [-1.77355014, -1.19049131,  1.92520594,  0.91837402]],
      shape=(7654, 4))

In [15]:
import torch
import torch.nn as nn

In [16]:
x_train_tensor = torch.tensor(x_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)

x_test_tensor = torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor = torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)



In [17]:
from torch.utils.data import DataLoader,TensorDataset

In [18]:
train_dataset = TensorDataset(x_train_tensor,y_train_tensor)
test_dataset = TensorDataset(x_test_tensor,y_test_tensor)

In [19]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32)

In [20]:
# building ann model


In [26]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model = nn.Sequential(
            # 1st layer
            nn.Linear(x_train.shape[1],6),
            nn.ReLU(),

            # 2nd layer 
            nn.Linear(6,6),
            nn.ReLU(),

            # output layer
            nn.Linear(6,1)
        )

    def forward(self,x):
        return self.model(x)


        

        

    

In [27]:
import torch.optim as optim
model = ANN()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())


training

In [28]:
epochs = 100
train_loss = []
valid_loss = []
best_val_loss=float("inf")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        yb = yb.view(-1, 1)  # ensure shape matches model output

        optimizer.zero_grad()
        output = model(xb)
        loss = criterion(output, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_loader)
    train_loss.append(epoch_train_loss)

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for xb, yb in test_loader:
            yb = yb.view(-1, 1)
            output = model(xb)
            loss = criterion(output, yb)
            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss / len(test_loader)
    valid_loss.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss:.4f} & val loss = {epoch_val_loss:.4f}")

    if epoch_val_loss<best_val_loss:
        best_val_loss= epoch_val_loss
        torch.save(model.state_dict(),"best_model.pt")

epoch 1/100 ==> train loss = 206538.8697 & val loss = 206408.8384
epoch 2/100 ==> train loss = 206347.1346 & val loss = 206191.0507
epoch 3/100 ==> train loss = 206065.4001 & val loss = 205973.4177
epoch 4/100 ==> train loss = 205868.5333 & val loss = 205755.9865
epoch 5/100 ==> train loss = 205661.0616 & val loss = 205538.6982
epoch 6/100 ==> train loss = 205441.7772 & val loss = 205321.5265
epoch 7/100 ==> train loss = 205208.7909 & val loss = 205104.5232
epoch 8/100 ==> train loss = 205057.6839 & val loss = 204887.6637
epoch 9/100 ==> train loss = 204776.2677 & val loss = 204670.8914
epoch 10/100 ==> train loss = 204563.1242 & val loss = 204454.3454
epoch 11/100 ==> train loss = 204356.1783 & val loss = 204237.8847
epoch 12/100 ==> train loss = 204127.7099 & val loss = 204021.5951
epoch 13/100 ==> train loss = 203965.3877 & val loss = 203805.3929
epoch 14/100 ==> train loss = 203714.2557 & val loss = 203589.3054
epoch 15/100 ==> train loss = 203535.7284 & val loss = 203373.3607
epoc

In [30]:
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

evaluation

In [31]:
model.eval()
with torch.no_grad():
    train_pred =model(x_train_tensor)
    test_pred = model(x_test_tensor)
    train_mse_loss = criterion(train_pred,y_train_tensor)
    test_mse_loss = criterion(test_pred,y_test_tensor)
    print(train_mse_loss.item())
    print(test_mse_loss.item())

185453.71875
185453.71875


In [32]:
from sklearn.metrics import r2_score
print(r2_score(y_test,test_pred))

ValueError: Found input variables with inconsistent numbers of samples: [1914, 7654]